In [ ]:
# ================================================================
# CELL 1 / 2 — PATH-GFP focused publication ablations + self-contained runtime
#
# R3-PATH = Projected Antithetic Transition-tangent Homotopy.
# Frozen-method confirmation pipeline. Default: repaired GFP versus R3,
# three focused tasks, three seeds, 1M steps, two GPUs, 100 final episodes.
# This is the minimum evidence gate for rapid AAMAS-style paper drafting,
# not a full Machine Learning / Neural Networks journal matrix.
# ================================================================
from __future__ import annotations

import importlib.util
import json
import os
import py_compile
import re
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

# ----------------------- USER CONFIG -----------------------------
# anchor6: six tasks individually reported by GFP/FAC.
# supported12: expand only after the anchor6 gate is positive.
# legacy_mujoco_diag3: not individually reported by GFP; diagnostic only.
# toy_smoke: engineering validation.
STAGE = "custom"
CUSTOM_DATASETS = ["antsoccer-arena-navigate-singletask-task4-v0", "scene-play-singletask-task2-v0", "pen-cloned-v1"]

METHODS = ["cats_gfp_r3_path", "cats_gfp_r3_no_projection", "cats_gfp_r3_no_homotopy", "cats_gfp_r3_gaussian_probe", "cats_gfp_r3_uniform_probe"]
SEEDS = [0, 1, 2]
STEPS = 1_000_000               # paper-level 1M budget
EVAL_EVERY = 250_000
EVAL_EPISODES = 20
FINAL_EVAL_EPISODES = 100
EVAL_SEED = 1024
LOG_EVERY = 10_000
SAVE_EVERY = 50_000

GPUS = [0, 1]
WORKERS_PER_GPU = 1
REQUIRE_GPU = True
MUJOCO_GL = "egl"
SERIAL_DATASET_PREFLIGHT = True

OUTPUT_ROOT = Path(f"./PATH_GFP_FOCUSED_ABLATIONS_{STEPS//1000}K_{len(SEEDS)}SEED_RESULTS").resolve()
RESUME = True
STOP_ON_FAILURE = False
MAX_JOB_RETRIES = 1
AUTO_INSTALL_MISSING = True

# Publication-confirmation statistics.
BOOTSTRAP_REPLICATES = 5_000
BOOTSTRAP_SEED = 20260820
CONFIDENCE_LEVEL = 0.95
TASK_MEANINGFUL_DELTA = 3.0
TASK_SEVERE_DELTA = -5.0
TIE_BAND = 1.0
MAX_RUNTIME_RATIO = 1.60

# Repaired, equation-faithful PyTorch GFP port.
HIDDEN = 512
DEPTH = 4
TIME_DIM = 64
LR = 3e-4
TAU = 0.005

# R3-PATH intervention. It is calibrated under pure GFP, active only in a
# middle homotopy window, and exactly zero in the late stage.
PROBE_RADIUS = 0.08
PROBE_STEPS = 100_000
PROBE_BURN_IN = 2_000
CALIBRATION_START = 100_000
CALIBRATION_END = 200_000
ACTIVE_END = 733_333
BUDGET_MULTIPLIER = 1.25
BUDGET_EMA_BETA = 0.99
MIN_TANGENT_BUDGET = 1e-4
LAMBDA_MAX = 0.03
AUXILIARY_GRAD_RATIO = 0.25
SAFETY_MARGIN = 0.0
SAFETY_TEMPERATURE = 0.05
MAX_RESIDUAL = 1.5
MIN_PROBE_RMS = 1e-4
MIN_GATE_MASS = 0.05
PROBE_DENOMINATOR_FLOOR = 1e-6

ANCHOR6 = ['antmaze-large-navigate-singletask-task1-v0', 'humanoidmaze-medium-navigate-singletask-task1-v0', 'antsoccer-arena-navigate-singletask-task4-v0', 'cube-single-play-singletask-task2-v0', 'scene-play-singletask-task2-v0', 'pen-cloned-v1']
SUPPORTED12 = ['antmaze-large-navigate-singletask-task1-v0', 'antmaze-giant-navigate-singletask-task1-v0', 'humanoidmaze-medium-navigate-singletask-task1-v0', 'humanoidmaze-large-navigate-singletask-task1-v0', 'antsoccer-arena-navigate-singletask-task4-v0', 'cube-single-play-singletask-task2-v0', 'cube-double-play-singletask-task2-v0', 'scene-play-singletask-task2-v0', 'puzzle-3x3-play-singletask-task4-v0', 'puzzle-4x4-play-singletask-task4-v0', 'pen-cloned-v1', 'antmaze-large-diverse-v2']
LEGACY_MUJOCO_DIAG3 = ['hopper-medium-replay-v2', 'halfcheetah-medium-replay-v2', 'walker2d-medium-replay-v2']
TOY_SMOKE = ['toy-transition-v0']

if STAGE == "anchor6":
    DATASETS = ANCHOR6
elif STAGE == "supported12":
    DATASETS = SUPPORTED12
elif STAGE == "legacy_mujoco_diag3":
    DATASETS = LEGACY_MUJOCO_DIAG3
elif STAGE == "toy_smoke":
    DATASETS = TOY_SMOKE
elif STAGE == "custom":
    if not CUSTOM_DATASETS:
        raise ValueError("CUSTOM_DATASETS must be non-empty when STAGE='custom'.")
    DATASETS = list(CUSTOM_DATASETS)
else:
    raise ValueError(f"Unknown STAGE={STAGE!r}")

if STAGE != "toy_smoke" and STEPS != 1_000_000:
    raise ValueError("This open-source notebook is configured for the 1M paper-level budget.")
ABLATION_METHODS = {
    "cats_gfp_r3_path",
    "cats_gfp_r3_no_projection",
    "cats_gfp_r3_no_homotopy",
    "cats_gfp_r3_gaussian_probe",
    "cats_gfp_r3_uniform_probe",
}
if set(METHODS) != ABLATION_METHODS:
    raise ValueError(f"Focused ablation notebook requires exactly: {sorted(ABLATION_METHODS)}")
if STAGE != "toy_smoke" and len(set(SEEDS)) < 3:
    raise ValueError("Publication confirmation requires at least three distinct training seeds.")
if len(set(SEEDS)) != len(SEEDS):
    raise ValueError("SEEDS must be distinct.")
if STAGE != "toy_smoke" and not (0 <= CALIBRATION_START < CALIBRATION_END < ACTIVE_END < STEPS):
    raise ValueError("Require 0 <= CALIBRATION_START < CALIBRATION_END < ACTIVE_END < STEPS.")

BACKBONE_SUPPORT = [{'dataset': 'hopper-medium-replay-v2', 'suite': 'd4rl', 'gfp_reported': False, 'gfp_score': None, 'fac_reported': True, 'fac_score': 99.1, 'fac_std': 0.9, 'default_role': 'legacy_mujoco_diag3', 'formal_gate_weight': 0.0, 'note': 'Not individually reported by GFP; reported by FAC.'}, {'dataset': 'halfcheetah-medium-replay-v2', 'suite': 'd4rl', 'gfp_reported': False, 'gfp_score': None, 'fac_reported': True, 'fac_score': 55.4, 'fac_std': 2.7, 'default_role': 'legacy_mujoco_diag3', 'formal_gate_weight': 0.0, 'note': 'Not individually reported by GFP; reported by FAC.'}, {'dataset': 'walker2d-medium-replay-v2', 'suite': 'd4rl', 'gfp_reported': False, 'gfp_score': None, 'fac_reported': True, 'fac_score': 83.0, 'fac_std': 5.8, 'default_role': 'legacy_mujoco_diag3', 'formal_gate_weight': 0.0, 'note': 'Not individually reported by GFP; reported by FAC.'}, {'dataset': 'pen-cloned-v1', 'suite': 'd4rl', 'gfp_reported': True, 'gfp_score': 77.1, 'gfp_std': 10.4, 'fac_reported': True, 'fac_score': 103.2, 'fac_std': 11.1, 'default_role': 'anchor6', 'formal_gate_weight': 1.0, 'note': 'Reported by both; R1 reached 85.96 at 300K.'}, {'dataset': 'antmaze-large-diverse-v2', 'suite': 'd4rl', 'gfp_reported': True, 'gfp_score': 84.1, 'gfp_std': 5.4, 'fac_reported': True, 'fac_score': 88.0, 'fac_std': 6.0, 'default_role': 'supported12', 'formal_gate_weight': 1.0, 'note': 'Reported by both; R1 stayed at zero.'}, {'dataset': 'antmaze-large-navigate-singletask-task1-v0', 'suite': 'ogbench', 'gfp_reported': True, 'gfp_score': 95.4, 'gfp_std': 0.8, 'fac_reported': True, 'fac_score': 94.0, 'fac_std': 3.0, 'default_role': 'anchor6', 'formal_gate_weight': 1.0, 'note': 'Strong GFP/FAC task; backbone reproduction sentinel.'}, {'dataset': 'antmaze-giant-navigate-singletask-task1-v0', 'suite': 'ogbench', 'gfp_reported': True, 'gfp_score': 12.6, 'gfp_std': 15.1, 'fac_reported': True, 'fac_score': 6.5, 'fac_std': 5.2, 'default_role': 'supported12', 'formal_gate_weight': 1.0, 'note': 'Both report low/high-variance performance; not a first-stage rejection task.'}, {'dataset': 'humanoidmaze-medium-navigate-singletask-task1-v0', 'suite': 'ogbench', 'gfp_reported': True, 'gfp_score': 83.5, 'gfp_std': 3.7, 'fac_reported': True, 'fac_score': 71.5, 'fac_std': 14.7, 'default_role': 'anchor6', 'formal_gate_weight': 1.0, 'note': 'Strong GFP task; backbone reproduction sentinel.'}, {'dataset': 'humanoidmaze-large-navigate-singletask-task1-v0', 'suite': 'ogbench', 'gfp_reported': True, 'gfp_score': 57.2, 'gfp_std': 23.7, 'fac_reported': True, 'fac_score': 15.0, 'fac_std': 19.8, 'default_role': 'supported12', 'formal_gate_weight': 1.0, 'note': 'Reported by both with high variance.'}, {'dataset': 'antsoccer-arena-navigate-singletask-task4-v0', 'suite': 'ogbench', 'gfp_reported': True, 'gfp_score': 40.2, 'gfp_std': 4.2, 'fac_reported': True, 'fac_score': 53.0, 'fac_std': 5.6, 'default_role': 'anchor6', 'formal_gate_weight': 1.0, 'note': 'Reported by both; R1 had a corrupt cache technical failure.'}, {'dataset': 'cube-single-play-singletask-task2-v0', 'suite': 'ogbench', 'gfp_reported': True, 'gfp_score': 99.4, 'gfp_std': 0.7, 'fac_reported': True, 'fac_score': 100.0, 'fac_std': 0.0, 'default_role': 'anchor6', 'formal_gate_weight': 1.0, 'note': 'Near-saturated backbone sentinel.'}, {'dataset': 'cube-double-play-singletask-task2-v0', 'suite': 'ogbench', 'gfp_reported': True, 'gfp_score': 53.3, 'gfp_std': 8.9, 'fac_reported': True, 'fac_score': 37.5, 'fac_std': 10.0, 'default_role': 'supported12', 'formal_gate_weight': 1.0, 'note': 'Reported by both; useful compositional manipulation expansion task.'}, {'dataset': 'scene-play-singletask-task2-v0', 'suite': 'ogbench', 'gfp_reported': True, 'gfp_score': 89.0, 'gfp_std': 4.1, 'fac_reported': True, 'fac_score': 100.0, 'fac_std': 0.0, 'default_role': 'anchor6', 'formal_gate_weight': 1.0, 'note': 'Strong manipulation sentinel; R1 collapsed from 20 to 5.'}, {'dataset': 'puzzle-3x3-play-singletask-task4-v0', 'suite': 'ogbench', 'gfp_reported': True, 'gfp_score': 5.4, 'gfp_std': 2.1, 'fac_reported': True, 'fac_score': 100.0, 'fac_std': 0.0, 'default_role': 'supported12', 'formal_gate_weight': 1.0, 'note': 'Backbone discriminator: GFP weak, FAC strong. Do not use as a hard GFP-anchor gate.'}, {'dataset': 'puzzle-4x4-play-singletask-task4-v0', 'suite': 'ogbench', 'gfp_reported': True, 'gfp_score': 17.2, 'gfp_std': 2.5, 'fac_reported': True, 'fac_score': 35.0, 'fac_std': 12.4, 'default_role': 'supported12', 'formal_gate_weight': 1.0, 'note': 'Reported by both; expansion task.'}]
GFP_PROFILES = {'antmaze-large-navigate-singletask-task1-v0': {'batch_size': 256, 'discount': 0.995, 'flow_steps': 10, 'q_agg': 'min', 'target_agg': 'actor', 'guidance_agg': 'min', 'alpha': 0.3, 'eta_temperature': 0.0001, 'profile_source': 'GFP official family profile'}, 'antmaze-giant-navigate-singletask-task1-v0': {'batch_size': 1024, 'discount': 0.995, 'flow_steps': 10, 'q_agg': 'min', 'target_agg': 'actor', 'guidance_agg': 'min', 'alpha': 0.1, 'eta_temperature': 0.1, 'profile_source': 'GFP official family profile'}, 'humanoidmaze-medium-navigate-singletask-task1-v0': {'batch_size': 256, 'discount': 0.995, 'flow_steps': 10, 'q_agg': 'mean', 'target_agg': 'mean', 'guidance_agg': 'min', 'alpha': 0.3, 'eta_temperature': 0.001, 'profile_source': 'GFP official family profile'}, 'humanoidmaze-large-navigate-singletask-task1-v0': {'batch_size': 1024, 'discount': 0.999, 'flow_steps': 30, 'q_agg': 'mean', 'target_agg': 'actor', 'guidance_agg': 'min', 'alpha': 0.3, 'eta_temperature': 0.0001, 'profile_source': 'GFP official family profile'}, 'antsoccer-arena-navigate-singletask-task4-v0': {'batch_size': 256, 'discount': 0.995, 'flow_steps': 10, 'q_agg': 'mean', 'target_agg': 'actor', 'guidance_agg': 'min', 'alpha': 0.1, 'eta_temperature': 0.01, 'profile_source': 'GFP official family profile'}, 'cube-single-play-singletask-task2-v0': {'batch_size': 256, 'discount': 0.99, 'flow_steps': 10, 'q_agg': 'mean', 'target_agg': 'mean', 'guidance_agg': 'actor', 'alpha': 10.0, 'eta_temperature': 0.1, 'profile_source': 'GFP official family profile'}, 'cube-double-play-singletask-task2-v0': {'batch_size': 256, 'discount': 0.99, 'flow_steps': 10, 'q_agg': 'mean', 'target_agg': 'mean', 'guidance_agg': 'actor', 'alpha': 1.0, 'eta_temperature': 0.01, 'profile_source': 'GFP official family profile'}, 'scene-play-singletask-task2-v0': {'batch_size': 256, 'discount': 0.99, 'flow_steps': 10, 'q_agg': 'mean', 'target_agg': 'actor', 'guidance_agg': 'actor', 'alpha': 10.0, 'eta_temperature': 0.001, 'profile_source': 'GFP official family profile'}, 'puzzle-3x3-play-singletask-task4-v0': {'batch_size': 256, 'discount': 0.99, 'flow_steps': 10, 'q_agg': 'mean', 'target_agg': 'actor', 'guidance_agg': 'actor', 'alpha': 3.0, 'eta_temperature': 0.001, 'profile_source': 'GFP official family profile'}, 'puzzle-4x4-play-singletask-task4-v0': {'batch_size': 256, 'discount': 0.99, 'flow_steps': 10, 'q_agg': 'mean', 'target_agg': 'actor', 'guidance_agg': 'actor', 'alpha': 3.0, 'eta_temperature': 1e-05, 'profile_source': 'GFP official family profile'}, 'antmaze-large-diverse-v2': {'batch_size': 256, 'discount': 0.99, 'flow_steps': 10, 'q_agg': 'min', 'target_agg': 'actor', 'guidance_agg': 'actor', 'alpha': 0.03, 'eta_temperature': 1e-05, 'profile_source': 'GFP official task profile'}, 'pen-cloned-v1': {'batch_size': 256, 'discount': 0.99, 'flow_steps': 10, 'q_agg': 'mean', 'target_agg': 'actor', 'guidance_agg': 'actor', 'alpha': 3.0, 'eta_temperature': 1e-05, 'profile_source': 'GFP official task profile'}, 'halfcheetah-medium-replay-v2': {'batch_size': 1024, 'discount': 0.99, 'flow_steps': 10, 'q_agg': 'mean', 'target_agg': 'actor', 'guidance_agg': 'actor', 'alpha': 0.1, 'eta_temperature': 0.001, 'profile_source': 'heuristic nearest GFP Minari-medium profile; diagnostic only'}, 'hopper-medium-replay-v2': {'batch_size': 1024, 'discount': 0.99, 'flow_steps': 10, 'q_agg': 'mean', 'target_agg': 'actor', 'guidance_agg': 'actor', 'alpha': 3.0, 'eta_temperature': 0.001, 'profile_source': 'heuristic nearest GFP Minari-medium profile; diagnostic only'}, 'walker2d-medium-replay-v2': {'batch_size': 1024, 'discount': 0.99, 'flow_steps': 10, 'q_agg': 'mean', 'target_agg': 'actor', 'guidance_agg': 'actor', 'alpha': 0.3, 'eta_temperature': 0.01, 'profile_source': 'heuristic nearest GFP Minari-medium profile; diagnostic only'}, 'toy-transition-v0': {'batch_size': 64, 'discount': 0.99, 'flow_steps': 4, 'q_agg': 'mean', 'target_agg': 'actor', 'guidance_agg': 'actor', 'alpha': 1.0, 'eta_temperature': 0.01, 'profile_source': 'toy smoke profile'}}
RUNNER_SOURCE = 'from __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport math\nimport os\nimport random\nimport re\nimport time\nimport zipfile\nfrom contextlib import contextmanager\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Dict, Optional, Tuple\n\nimport numpy as np\nimport torch\nfrom torch import Tensor, nn\nimport torch.nn.functional as F\n\n\n# ============================================================\n# Utilities\n# ============================================================\n\ndef set_seed(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef atomic_json(path: Path, payload: Dict[str, Any]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True, default=float))\n    tmp.replace(path)\n\n\ndef append_csv(path: Path, row: Dict[str, Any]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    exists = path.exists()\n    with path.open("a", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=list(row.keys()))\n        if not exists:\n            writer.writeheader()\n        writer.writerow(row)\n\n\ndef soft_update(source: nn.Module, target: nn.Module, tau: float) -> None:\n    with torch.no_grad():\n        for p, tp in zip(source.parameters(), target.parameters()):\n            tp.data.mul_(1.0 - tau).add_(p.data, alpha=tau)\n\n\ndef grad_norm(parameters) -> float:\n    total = 0.0\n    for p in parameters:\n        if p.grad is not None:\n            total += float(p.grad.detach().square().sum())\n    return math.sqrt(total)\n\n\ndef finite_float(x: Any, default: float = float("nan")) -> float:\n    try:\n        v = float(x)\n        return v if math.isfinite(v) else default\n    except Exception:\n        return default\n\n\n@contextmanager\ndef frozen_parameters(module: nn.Module):\n    """Freeze parameter gradients while preserving input/action derivatives."""\n    params = list(module.parameters())\n    flags = [p.requires_grad for p in params]\n    try:\n        for p in params:\n            p.requires_grad_(False)\n        yield\n    finally:\n        for p, flag in zip(params, flags):\n            p.requires_grad_(flag)\n\n\ndef autograd_list(loss: Tensor, params, *, retain_graph: bool = False):\n    params = list(params)\n    grads = torch.autograd.grad(\n        loss,\n        params,\n        retain_graph=retain_graph,\n        create_graph=False,\n        allow_unused=True,\n    )\n    return [torch.zeros_like(p) if g is None else g for p, g in zip(params, grads)]\n\n\ndef list_dot(xs, ys) -> Tensor:\n    if not xs:\n        return torch.zeros(())\n    return sum((x * y).sum() for x, y in zip(xs, ys))\n\n\ndef list_norm(xs) -> Tensor:\n    return torch.sqrt(list_dot(xs, xs).clamp_min(0.0))\n\n\ndef assign_gradients(params, grads) -> None:\n    for p, g in zip(params, grads):\n        p.grad = g.detach().clone()\n\n\n# ============================================================\n# Official-equation-faithful GFP modules (PyTorch port)\n# ============================================================\n\nclass MLP(nn.Module):\n    """GELU MLP matching the released GFP/FQL architecture shape."""\n    def __init__(self, in_dim: int, out_dim: int, hidden: int, depth: int, *, layer_norm: bool):\n        super().__init__()\n        self.hidden = nn.ModuleList()\n        d = in_dim\n        for _ in range(depth):\n            self.hidden.append(nn.Linear(d, hidden))\n            d = hidden\n        self.output = nn.Linear(d, out_dim)\n        self.layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(depth)]) if layer_norm else None\n        self.reset_parameters()\n\n    def reset_parameters(self) -> None:\n        for module in [*self.hidden, self.output]:\n            nn.init.xavier_uniform_(module.weight)\n            nn.init.zeros_(module.bias)\n\n    def forward(self, x: Tensor) -> Tensor:\n        for i, layer in enumerate(self.hidden):\n            x = F.gelu(layer(x))\n            if self.layer_norms is not None:\n                x = self.layer_norms[i](x)\n        return self.output(x)\n\n\nclass SinusoidalTimeEmbedding(nn.Module):\n    def __init__(self, dim: int = 64):\n        super().__init__()\n        self.dim = int(dim)\n\n    def forward(self, t: Tensor) -> Tensor:\n        if t.ndim == 1:\n            t = t.unsqueeze(-1)\n        half = self.dim // 2\n        if half <= 1:\n            return t\n        freqs = torch.exp(\n            torch.arange(half, device=t.device, dtype=t.dtype)\n            * (-math.log(10000.0) / float(half - 1))\n        )\n        args = t * freqs\n        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)\n        if emb.shape[-1] < self.dim:\n            emb = F.pad(emb, (0, self.dim - emb.shape[-1]))\n        return emb\n\n\nclass OneStepActor(nn.Module):\n    """GFP one-step actor A(s,z). Output is clipped only where the released code clips it."""\n    def __init__(self, state_dim: int, action_dim: int, hidden: int, depth: int):\n        super().__init__()\n        self.action_dim = action_dim\n        self.net = MLP(state_dim + action_dim, action_dim, hidden, depth, layer_norm=False)\n\n    def forward(self, state: Tensor, noise: Tensor) -> Tensor:\n        return self.net(torch.cat([state, noise], dim=-1))\n\n\nclass FlowVelocity(nn.Module):\n    def __init__(self, state_dim: int, action_dim: int, hidden: int, depth: int, time_dim: int = 64):\n        super().__init__()\n        self.time_emb = SinusoidalTimeEmbedding(time_dim)\n        self.net = MLP(state_dim + action_dim + time_dim, action_dim, hidden, depth, layer_norm=False)\n\n    def forward(self, state: Tensor, action_t: Tensor, t: Tensor) -> Tensor:\n        return self.net(torch.cat([state, action_t, self.time_emb(t)], dim=-1))\n\n\nclass QHead(nn.Module):\n    def __init__(self, state_dim: int, action_dim: int, hidden: int, depth: int):\n        super().__init__()\n        self.net = MLP(state_dim + action_dim, 1, hidden, depth, layer_norm=True)\n\n    def forward(self, state: Tensor, action: Tensor) -> Tensor:\n        return self.net(torch.cat([state, action], dim=-1)).squeeze(-1)\n\n\nclass TwinCritic(nn.Module):\n    def __init__(self, state_dim: int, action_dim: int, hidden: int, depth: int):\n        super().__init__()\n        self.q1 = QHead(state_dim, action_dim, hidden, depth)\n        self.q2 = QHead(state_dim, action_dim, hidden, depth)\n\n    def forward(self, state: Tensor, action: Tensor) -> Tuple[Tensor, Tensor]:\n        return self.q1(state, action), self.q2(state, action)\n\n\ndef q_aggregate(critic: TwinCritic, state: Tensor, action: Tensor, rule: str) -> Tensor:\n    q1, q2 = critic(state, action)\n    if rule == "min":\n        return torch.minimum(q1, q2)\n    if rule == "mean":\n        return 0.5 * (q1 + q2)\n    if rule == "max":\n        return torch.maximum(q1, q2)\n    raise ValueError(f"unknown q aggregation: {rule}")\n\n\ndef value_options(\n    critic: TwinCritic,\n    obs: Tensor,\n    actor_actions: Optional[Tensor],\n    flow_actions: Optional[Tensor],\n    *,\n    rule: str,\n    q_rule: str,\n) -> Tensor:\n    if rule == "actor":\n        assert actor_actions is not None\n        return q_aggregate(critic, obs, actor_actions, q_rule)\n    if rule in {"vabc", "flow"}:\n        assert flow_actions is not None\n        return q_aggregate(critic, obs, flow_actions, q_rule)\n    assert actor_actions is not None and flow_actions is not None\n    qa = q_aggregate(critic, obs, actor_actions, q_rule)\n    qf = q_aggregate(critic, obs, flow_actions, q_rule)\n    if rule == "min":\n        return torch.minimum(qa, qf)\n    if rule == "mean":\n        return 0.5 * (qa + qf)\n    if rule == "max":\n        return torch.maximum(qa, qf)\n    raise ValueError(f"unknown value aggregation: {rule}")\n\n\ndef flow_sample(flow: FlowVelocity, state: Tensor, noise: Tensor, steps: int) -> Tensor:\n    action = noise\n    dt = 1.0 / float(steps)\n    # Released GFP uses Euler time i/K.\n    for i in range(int(steps)):\n        t = torch.full((*state.shape[:-1], 1), i / float(steps), device=state.device, dtype=state.dtype)\n        action = action + dt * flow(state, action, t)\n    return action.clamp(-1.0, 1.0)\n\n\n# ============================================================\n# Hyperchaotic Lorenz transition probes\n# ============================================================\n\nclass LorenzProbeBank:\n    """Bounded deterministic probes. No claim that the probe is environment noise."""\n    def __init__(\n        self,\n        *,\n        steps: int = 100_000,\n        burn_in: int = 2_000,\n        dt: float = 0.005,\n        sigma: float = 10.0,\n        rho: float = 28.0,\n        beta: float = 8.0 / 3.0,\n        eta: float = -1.0,\n        initial=(1.0, 1.0, 1.0, 0.5),\n        delay_lag: int = 7,\n        stride: int = 13,\n    ) -> None:\n        self.delay_lag = int(delay_lag)\n        self.stride = int(stride)\n        self.cursor = 0\n\n        def field(x):\n            xx, yy, zz, ww = x\n            return np.asarray([\n                sigma * (yy - xx),\n                xx * (rho - zz) - yy,\n                xx * yy - beta * zz,\n                -yy * zz + eta * ww,\n            ], np.float64)\n\n        x = np.asarray(initial, np.float64)\n        traj = np.empty((steps + burn_in, 4), np.float64)\n        for i in range(len(traj)):\n            k1 = field(x)\n            k2 = field(x + 0.5 * dt * k1)\n            k3 = field(x + 0.5 * dt * k2)\n            k4 = field(x + dt * k3)\n            x = x + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)\n            if not np.all(np.isfinite(x)):\n                raise FloatingPointError("Lorenz probe integration diverged")\n            traj[i] = x\n        traj = traj[burn_in:]\n        traj = (traj - traj.mean(0, keepdims=True)) / np.maximum(traj.std(0, keepdims=True), 1e-8)\n        scale = np.maximum(np.quantile(np.abs(traj), 0.995, axis=0, keepdims=True), 1e-8)\n        self.stream = np.clip(traj / scale, -1.0, 1.0).reshape(-1).astype(np.float32)\n\n    def sample(self, batch: int, dim: int, device: torch.device, dtype: torch.dtype) -> Tensor:\n        max_delay = (dim - 1) * self.delay_lag\n        available = len(self.stream) - max_delay - 1\n        starts = (self.cursor + self.stride * np.arange(batch, dtype=np.int64)) % available\n        offsets = self.delay_lag * np.arange(dim, dtype=np.int64)\n        vals = self.stream[starts[:, None] + offsets[None, :]]\n        self.cursor = int((self.cursor + self.stride * batch) % available)\n        return torch.as_tensor(vals, device=device, dtype=dtype)\n\n\nclass IIDProbeBank:\n    """Deterministic bounded probe stream used only for matched ablations."""\n    def __init__(self, *, kind: str, steps: int = 100_000, seed: int = 0, delay_lag: int = 7, stride: int = 13):\n        if kind not in {"gaussian", "uniform"}:\n            raise ValueError(kind)\n        self.delay_lag = int(delay_lag)\n        self.stride = int(stride)\n        self.cursor = 0\n        rng = np.random.default_rng(int(seed) + (9137 if kind == "gaussian" else 19139))\n        n = max(int(steps) * 4, 10_000)\n        if kind == "gaussian":\n            raw = rng.normal(size=n).astype(np.float64)\n            scale = max(float(np.quantile(np.abs(raw), 0.995)), 1e-8)\n            raw = np.clip(raw / scale, -1.0, 1.0)\n        else:\n            raw = rng.uniform(-1.0, 1.0, size=n)\n        self.stream = raw.astype(np.float32)\n\n    def sample(self, batch: int, dim: int, device: torch.device, dtype: torch.dtype) -> Tensor:\n        max_delay = (dim - 1) * self.delay_lag\n        available = len(self.stream) - max_delay - 1\n        starts = (self.cursor + self.stride * np.arange(batch, dtype=np.int64)) % available\n        offsets = self.delay_lag * np.arange(dim, dtype=np.int64)\n        vals = self.stream[starts[:, None] + offsets[None, :]]\n        self.cursor = int((self.cursor + self.stride * batch) % available)\n        return torch.as_tensor(vals, device=device, dtype=dtype)\n\n\n# ============================================================\n# Dataset loading\n# ============================================================\n\n@contextmanager\ndef dataset_cache_lock(name: str):\n    root = os.environ.get("CATS_DATASET_LOCK_DIR", "").strip()\n    if not root:\n        yield\n        return\n    root_path = Path(root)\n    root_path.mkdir(parents=True, exist_ok=True)\n    key = re.sub(r"-singletask-task\\d+-v0$", "", name)\n    key = re.sub(r"[^A-Za-z0-9_.-]+", "_", key)\n    fh = (root_path / (key + ".lock")).open("a+")\n    try:\n        try:\n            import fcntl\n            fcntl.flock(fh.fileno(), fcntl.LOCK_EX)\n        except Exception:\n            pass\n        yield\n    finally:\n        try:\n            import fcntl\n            fcntl.flock(fh.fileno(), fcntl.LOCK_UN)\n        except Exception:\n            pass\n        fh.close()\n\n\ndef detect_suite(name: str) -> str:\n    if name.startswith("toy-"):\n        return "toy"\n    if "singletask" in name or name.startswith("ogbench:"):\n        return "ogbench"\n    return "d4rl"\n\n\nclass ToyLinearEnv:\n    def __init__(self, state_dim=5, action_dim=2, horizon=40, seed=0):\n        self.state_dim = state_dim\n        self.action_dim = action_dim\n        self.horizon = horizon\n        self.rng = np.random.default_rng(seed)\n        self.A = (0.93 * np.eye(state_dim)).astype(np.float32)\n        self.B = self.rng.normal(0, 0.2, (state_dim, action_dim)).astype(np.float32)\n        self.goal = np.linspace(-0.4, 0.4, state_dim).astype(np.float32)\n        self.action_space = type("Box", (), {"low": -np.ones(action_dim, np.float32), "high": np.ones(action_dim, np.float32)})()\n        self.observation_space = type("Box", (), {"shape": (state_dim,)})()\n        self.spec = type("Spec", (), {"max_episode_steps": horizon})()\n        self.t = 0\n        self.s = np.zeros(state_dim, np.float32)\n\n    def reset(self, seed=None, **kwargs):\n        if seed is not None:\n            self.rng = np.random.default_rng(seed)\n        self.t = 0\n        self.s = self.rng.normal(0, 0.4, self.state_dim).astype(np.float32)\n        return self.s.copy(), {}\n\n    def step(self, action):\n        a = np.asarray(action, np.float32).clip(-1, 1)\n        ns = self.A @ self.s + self.B @ a + 0.01 * self.rng.normal(size=self.state_dim)\n        reward = -float(np.square(ns - self.goal).mean() + 0.02 * np.square(a).mean())\n        self.s = ns.astype(np.float32)\n        self.t += 1\n        truncated = self.t >= self.horizon\n        success = float(np.linalg.norm(self.s - self.goal) < 0.35)\n        return self.s.copy(), reward, False, truncated, {"success": success}\n\n    def close(self):\n        pass\n\n\ndef make_toy_dataset(seed=0, n=12_000):\n    env = ToyLinearEnv(seed=seed)\n    rows = {k: [] for k in ["observations", "actions", "rewards", "next_observations", "terminals", "masks"]}\n    s, _ = env.reset(seed=seed)\n    rng = np.random.default_rng(seed + 11)\n    for i in range(n):\n        a = np.tanh(-0.45 * s[:2] + 0.35 * rng.normal(size=2)).astype(np.float32)\n        ns, r, terminated, truncated, _ = env.step(a)\n        rows["observations"].append(s)\n        rows["actions"].append(a)\n        rows["rewards"].append(r)\n        rows["next_observations"].append(ns)\n        rows["terminals"].append(float(terminated or truncated))\n        rows["masks"].append(float(not terminated))\n        s = ns\n        if terminated or truncated:\n            s, _ = env.reset(seed=seed + i + 1)\n    data = {k: np.asarray(v, np.float32) for k, v in rows.items()}\n    split = int(0.9 * n)\n    return env, {k: v[:split] for k, v in data.items()}, {k: v[split:] for k, v in data.items()}\n\n\ndef _ogbench_cache_roots():\n    roots = []\n    explicit = os.environ.get("OGBENCH_DATA_DIR", "").strip()\n    if explicit:\n        roots.append(Path(explicit).expanduser())\n    roots.extend([\n        Path.home() / ".ogbench",\n        Path.home() / ".cache" / "ogbench",\n        Path.home() / "ogbench",\n        Path.cwd() / "datasets",\n        Path.cwd() / "data",\n    ])\n    out, seen = [], set()\n    for root in roots:\n        try:\n            key = str(root.resolve())\n        except Exception:\n            key = str(root)\n        if key not in seen:\n            out.append(root)\n            seen.add(key)\n    return out\n\n\ndef _matches_ogbench_family(path: Path, name: str) -> bool:\n    family = name.split("-singletask")[0]\n    tokens = {\n        family.lower(),\n        family.lower().replace("-", "_"),\n        family.lower().replace("_", "-"),\n    }\n    hay = str(path).lower().replace("\\\\", "/")\n    hay_u = hay.replace("-", "_")\n    return any(t in hay or t.replace("-", "_") in hay_u for t in tokens)\n\n\ndef _archive_is_valid(path: Path) -> bool:\n    try:\n        if path.suffix.lower() == ".npz":\n            obj = np.load(path, allow_pickle=False)\n            try:\n                _ = obj.files\n                if obj.files:\n                    # Force one array header/read; truncated downloads can pass ZipFile construction.\n                    _ = np.asarray(obj[obj.files[0]]).shape\n            finally:\n                obj.close()\n            return True\n        if path.suffix.lower() == ".zip":\n            with zipfile.ZipFile(path) as zf:\n                return zf.testzip() is None\n    except (zipfile.BadZipFile, EOFError, OSError, ValueError):\n        return False\n    return True\n\n\ndef _purge_corrupt_ogbench_archives(name: str) -> int:\n    """Remove only matching, invalid .npz/.zip cache files."""\n    removed = 0\n    for root in _ogbench_cache_roots():\n        if not root.exists():\n            continue\n        for pattern in ("*.npz", "*.zip"):\n            for path in root.rglob(pattern):\n                if not _matches_ogbench_family(path, name):\n                    continue\n                if _archive_is_valid(path):\n                    continue\n                try:\n                    path.unlink()\n                    removed += 1\n                    print(f"[cache repair] removed corrupt OGBench cache file: {path}", flush=True)\n                except OSError as exc:\n                    print(f"[cache repair] could not remove {path}: {exc}", flush=True)\n    return removed\n\n\ndef _ogbench_version_tuple():\n    try:\n        from importlib import metadata\n        text = metadata.version("ogbench")\n    except Exception:\n        return (0, 0, 0), "unknown"\n    nums = [int(x) for x in re.findall(r"\\\\d+", text)[:3]]\n    while len(nums) < 3:\n        nums.append(0)\n    return tuple(nums), text\n\n\ndef load_raw_dataset(name: str, seed: int):\n    suite = detect_suite(name)\n    if suite == "toy":\n        env, train, val = make_toy_dataset(seed=seed)\n        return env, train, val, suite\n    with dataset_cache_lock(name):\n        if suite == "ogbench":\n            import ogbench\n            env_name = name.split(":", 1)[1] if name.startswith("ogbench:") else name\n            version_tuple, version_text = _ogbench_version_tuple()\n            # Official GFP pins ogbench<1.2. Under that protocol the keyword\n            # success_timing is not accepted, so we do not pass it.\n            kwargs = dict(compact_dataset=False)\n            if version_tuple >= (1, 2, 0):\n                kwargs["success_timing"] = "post"\n            print(f"[ogbench] version={version_text}; env={env_name}; kwargs={kwargs}", flush=True)\n\n            def make_ogbench():\n                return ogbench.make_env_and_datasets(env_name, **kwargs)\n\n            try:\n                env, train, val = make_ogbench()\n            except (zipfile.BadZipFile, EOFError, OSError, ValueError) as exc:\n                removed = _purge_corrupt_ogbench_archives(env_name)\n                print(\n                    f"[cache repair] {type(exc).__name__} for {env_name}; removed={removed}; retrying once",\n                    flush=True,\n                )\n                env, train, val = make_ogbench()\n            train = {k: np.asarray(v) for k, v in train.items() if isinstance(v, (np.ndarray, list, tuple))}\n            val = {k: np.asarray(v) for k, v in val.items() if isinstance(v, (np.ndarray, list, tuple))}\n            return env, train, val, suite\n        import gym\n        import d4rl  # noqa: F401\n        env = gym.make(name)\n        data = {k: np.asarray(v) for k, v in d4rl.qlearning_dataset(env).items()}\n        return env, data, None, suite\n\n\n@dataclass\nclass TensorBatch:\n    state: Tensor\n    action: Tensor\n    reward: Tensor\n    next_state: Tensor\n    mask: Tensor\n    terminal: Tensor\n\n\n@dataclass\nclass DatasetArrays:\n    state: np.ndarray\n    action: np.ndarray\n    reward: np.ndarray\n    next_state: np.ndarray\n    mask: np.ndarray\n    terminal: np.ndarray\n    action_low: np.ndarray\n    action_high: np.ndarray\n    transition_rms: np.ndarray\n    name: str\n    suite: str\n\n    @property\n    def size(self):\n        return int(len(self.state))\n\n    def sample(self, batch_size: int, device: torch.device) -> TensorBatch:\n        idx = np.random.randint(0, self.size, size=int(batch_size))\n        def tt(x):\n            return torch.as_tensor(x[idx], device=device, dtype=torch.float32)\n        return TensorBatch(tt(self.state), tt(self.action), tt(self.reward), tt(self.next_state), tt(self.mask), tt(self.terminal))\n\n\ndef _action_bounds(env, action_dim: int):\n    low = np.asarray(getattr(env.action_space, "low", -np.ones(action_dim)), np.float32).reshape(-1)\n    high = np.asarray(getattr(env.action_space, "high", np.ones(action_dim)), np.float32).reshape(-1)\n    low = np.where(np.isfinite(low), low, -1.0).astype(np.float32)\n    high = np.where(np.isfinite(high), high, 1.0).astype(np.float32)\n    return low, high\n\n\ndef normalize_action(action, low, high):\n    span = np.maximum(high - low, 1e-6)\n    return np.clip(2.0 * (action - low) / span - 1.0, -1.0, 1.0).astype(np.float32)\n\n\ndef denormalize_action(action, low, high):\n    action = np.asarray(action)\n    return (low + 0.5 * (action + 1.0) * (high - low)).clip(low, high).astype(np.float32)\n\n\ndef prepare_dataset(env, raw: Dict[str, np.ndarray], suite: str, name: str) -> DatasetArrays:\n    state = np.asarray(raw["observations"], np.float32)\n    action = np.asarray(raw["actions"], np.float32)\n    reward = np.asarray(raw["rewards"], np.float32).reshape(-1, 1)\n    next_state = np.asarray(raw["next_observations"], np.float32)\n    terminal = np.asarray(raw.get("terminals", np.zeros(len(state))), np.float32).reshape(-1, 1)\n    if suite == "ogbench" and "masks" in raw:\n        mask = np.asarray(raw["masks"], np.float32).reshape(-1, 1)\n    else:\n        mask = (1.0 - terminal).astype(np.float32)\n    if suite == "d4rl" and name.startswith("antmaze-"):\n        reward = reward - 1.0\n    low, high = _action_bounds(env, action.shape[-1])\n    action = normalize_action(action, low, high)\n    transition_rms = np.sqrt(np.mean(np.square(next_state - state), axis=0) + 1e-6).astype(np.float32)\n    return DatasetArrays(\n        state=state,\n        action=action,\n        reward=reward,\n        next_state=next_state,\n        mask=mask,\n        terminal=terminal,\n        action_low=low,\n        action_high=high,\n        transition_rms=transition_rms,\n        name=name,\n        suite=suite,\n    )\n\n\n# ============================================================\n# Evaluation\n# ============================================================\n\ndef env_reset(env, seed: int):\n    try:\n        out = env.reset(seed=seed)\n    except TypeError:\n        try:\n            env.seed(seed)\n        except Exception:\n            pass\n        out = env.reset()\n    return out if isinstance(out, tuple) else (out, {})\n\n\ndef env_step(env, action):\n    out = env.step(action)\n    if len(out) == 5:\n        obs, reward, terminated, truncated, info = out\n    else:\n        obs, reward, done, info = out\n        terminated, truncated = bool(done), False\n    return obs, float(reward), bool(terminated), bool(truncated), info\n\n\ndef evaluate_policy(env, ds: DatasetArrays, actor: OneStepActor, *, episodes: int, seed: int, device: torch.device):\n    actor_was = actor.training\n    actor.eval()\n    returns, successes = [], []\n    rng = np.random.default_rng(seed + 19)\n    with torch.no_grad():\n        for ep in range(int(episodes)):\n            obs, _ = env_reset(env, seed + ep)\n            total, success, done, guard = 0.0, 0.0, False, 0\n            max_steps = int(getattr(getattr(env, "spec", None), "max_episode_steps", 0) or 10_000)\n            while not done and guard < max_steps + 5:\n                s = torch.as_tensor(np.asarray(obs, np.float32), device=device).unsqueeze(0)\n                z = torch.as_tensor(rng.normal(size=(1, actor.action_dim)), device=device, dtype=torch.float32)\n                a_norm = actor(s, z).clamp(-1.0, 1.0)[0].cpu().numpy()\n                a = denormalize_action(a_norm, ds.action_low, ds.action_high)\n                obs, reward, terminated, truncated, info = env_step(env, a)\n                total += reward\n                done = terminated or truncated\n                guard += 1\n                if isinstance(info, dict):\n                    success = max(success, finite_float(info.get("success", 0.0), 0.0))\n                if ds.suite == "ogbench" and terminated:\n                    success = max(success, 1.0)\n                if ds.suite == "d4rl" and ds.name.startswith("antmaze-") and reward > 0:\n                    success = max(success, 1.0)\n            returns.append(total)\n            successes.append(success)\n    actor.train(actor_was)\n    raw_return = float(np.mean(returns))\n    if ds.suite == "d4rl" and hasattr(env, "get_normalized_score"):\n        try:\n            score = float(env.get_normalized_score(raw_return)) * 100.0\n        except Exception:\n            score = raw_return\n    elif ds.suite == "ogbench":\n        score = float(np.mean(successes)) * 100.0\n    else:\n        score = raw_return\n    return {"score": score, "raw_return": raw_return, "success_rate": float(np.mean(successes)) * 100.0}\n\n\n# ============================================================\n# Training\n# ============================================================\n\ndef _pulse_schedule(step: int, start: int, end: int, maximum: float) -> float:\n    if step <= start or step >= end or end <= start:\n        return 0.0\n    u = (step - start) / float(end - start)\n    return float(maximum * math.sin(math.pi * u) ** 2)\n\n\ndef _huber_positive(x: Tensor) -> Tensor:\n    """Huber penalty for x>=0 with unit transition."""\n    return torch.where(x <= 1.0, 0.5 * x.square(), x - 0.5)\n\n\ndef _critic_twin_values(critic: TwinCritic, state: Tensor, action: Tensor):\n    with frozen_parameters(critic):\n        return critic(state, action)\n\n\ndef train(job: Dict[str, Any]):\n    name = job["dataset"]\n    method = job["method"]\n    seed = int(job["seed"])\n    output = Path(job["output"])\n    output.mkdir(parents=True, exist_ok=True)\n    atomic_json(output / "config.json", job)\n\n    set_seed(seed)\n    if not torch.cuda.is_available() and job.get("require_gpu", True):\n        raise RuntimeError("CUDA is required but unavailable")\n    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")\n\n    env, raw, _val, suite = load_raw_dataset(name, seed)\n    ds = prepare_dataset(env, raw, suite, name)\n    state_dim, action_dim = ds.state.shape[-1], ds.action.shape[-1]\n    profile = dict(job["profile"])\n    atomic_json(output / "resolved_profile.json", profile)\n\n    hidden = int(job["hidden"])\n    depth = int(job["depth"])\n    actor = OneStepActor(state_dim, action_dim, hidden, depth).to(device)\n    flow = FlowVelocity(state_dim, action_dim, hidden, depth, int(job["time_dim"])).to(device)\n    critic = TwinCritic(state_dim, action_dim, hidden, depth).to(device)\n    target_critic = TwinCritic(state_dim, action_dim, hidden, depth).to(device)\n    target_critic.load_state_dict(critic.state_dict())\n    target_critic.requires_grad_(False)\n\n    # A single Adam over disjoint parameter blocks matches the released optimizer\n    # organization. We manually assign module-wise gradients to reproduce the\n    # released stop-gradient paths and prevent actor Q loss from updating critic.\n    optimizer = torch.optim.Adam(\n        list(actor.parameters()) + list(flow.parameters()) + list(critic.parameters()),\n        lr=float(job["lr"]),\n    )\n    actor_params = list(actor.parameters())\n    flow_params = list(flow.parameters())\n    critic_params = list(critic.parameters())\n\n    if method == "cats_gfp_r3_gaussian_probe":\n        probe = IIDProbeBank(kind="gaussian", steps=int(job["probe_steps"]), seed=seed)\n    elif method == "cats_gfp_r3_uniform_probe":\n        probe = IIDProbeBank(kind="uniform", steps=int(job["probe_steps"]), seed=seed)\n    else:\n        probe = LorenzProbeBank(steps=int(job["probe_steps"]), burn_in=int(job["probe_burn_in"]))\n    transition_rms = torch.as_tensor(ds.transition_rms, device=device, dtype=torch.float32)\n\n    steps = int(job["steps"])\n    batch_size = int(profile["batch_size"])\n    discount = float(profile["discount"])\n    flow_steps = int(profile["flow_steps"])\n    q_rule = str(profile["q_agg"])\n    target_rule = str(profile["target_agg"])\n    guidance_rule = str(profile["guidance_agg"])\n    alpha = float(profile["alpha"])\n    eta = float(profile["eta_temperature"])\n    tau = float(job["tau"])\n\n    allowed_methods = {\n        "cats_gfp_r3_path",\n        "cats_gfp_r3_no_projection",\n        "cats_gfp_r3_no_homotopy",\n        "cats_gfp_r3_gaussian_probe",\n        "cats_gfp_r3_uniform_probe",\n    }\n    if method not in allowed_methods:\n        raise ValueError(f"Unsupported ablation method: {method}")\n    r3 = True\n    projection_enabled = method != "cats_gfp_r3_no_projection"\n    persistent_auxiliary = method == "cats_gfp_r3_no_homotopy"\n    calibration_start = int(job["calibration_start"])\n    calibration_end = int(job["calibration_end"])\n    active_end = int(job["active_end"])\n    probe_radius = float(job["probe_radius"])\n    budget_multiplier = float(job["budget_multiplier"])\n    budget_ema_beta = float(job["budget_ema_beta"])\n    min_tangent_budget = float(job["min_tangent_budget"])\n    lambda_max = float(job["lambda_max"]) if r3 else 0.0\n    auxiliary_grad_ratio = float(job["auxiliary_grad_ratio"])\n    safety_margin = float(job["safety_margin"])\n    safety_temperature = float(job["safety_temperature"])\n    max_residual = float(job["max_residual"])\n    min_probe_rms = float(job["min_probe_rms"])\n    min_gate_mass = float(job["min_gate_mass"])\n    probe_denominator_floor = float(job["probe_denominator_floor"])\n\n    budget_q75_ema = float("nan")\n    tangent_budget = float("nan")\n    budget_frozen = False\n\n    metrics_path = output / "metrics.csv"\n    train_metrics_path = output / "training_metrics.csv"\n    checkpoint_path = output / "checkpoint.pt"\n    start_step = 0\n\n    if job.get("resume", True) and checkpoint_path.exists():\n        ckpt = torch.load(checkpoint_path, map_location=device)\n        actor.load_state_dict(ckpt["actor"])\n        flow.load_state_dict(ckpt["flow"])\n        critic.load_state_dict(ckpt["critic"])\n        target_critic.load_state_dict(ckpt["target_critic"])\n        optimizer.load_state_dict(ckpt["optimizer"])\n        start_step = int(ckpt["step"])\n        probe.cursor = int(ckpt.get("probe_cursor", 0))\n        budget_q75_ema = float(ckpt.get("budget_q75_ema", float("nan")))\n        tangent_budget = float(ckpt.get("tangent_budget", float("nan")))\n        budget_frozen = bool(ckpt.get("budget_frozen", False))\n        print(f"[resume] {name} {method} from step {start_step}", flush=True)\n\n    def save_checkpoint(step: int):\n        tmp = checkpoint_path.with_suffix(".tmp")\n        torch.save({\n            "actor": actor.state_dict(),\n            "flow": flow.state_dict(),\n            "critic": critic.state_dict(),\n            "target_critic": target_critic.state_dict(),\n            "optimizer": optimizer.state_dict(),\n            "step": step,\n            "probe_cursor": probe.cursor,\n            "budget_q75_ema": budget_q75_ema,\n            "tangent_budget": tangent_budget,\n            "budget_frozen": budget_frozen,\n        }, tmp)\n        tmp.replace(checkpoint_path)\n\n    eval_every = int(job["eval_every"])\n    eval_episodes = int(job["eval_episodes"])\n    final_eval_episodes = int(job["final_eval_episodes"])\n    eval_seed = int(job["eval_seed"])\n    log_every = int(job["log_every"])\n    save_every = int(job["save_every"])\n    t0 = time.time()\n    last_diag: Dict[str, Any] = {}\n    last_eval_result: Optional[Dict[str, Any]] = None\n\n    for step in range(start_step + 1, steps + 1):\n        batch = ds.sample(batch_size, device)\n        z = torch.randn((batch_size, action_dim), device=device)\n\n        # ----- Critic target, released GFP aggregation semantics -----\n        with torch.no_grad():\n            next_actor = None if target_rule in {"vabc", "flow"} else actor(batch.next_state, z).clamp(-1.0, 1.0)\n            # Released default target_use_next=False for the VaBC branch.\n            flow_condition = batch.state\n            next_flow = None if target_rule == "actor" else flow_sample(flow, flow_condition, z, flow_steps)\n            next_q = value_options(\n                target_critic,\n                batch.next_state,\n                next_actor,\n                next_flow,\n                rule=target_rule,\n                q_rule=q_rule,\n            )\n            target_q = batch.reward.squeeze(-1) + discount * batch.mask.squeeze(-1) * next_q\n        q1, q2 = critic(batch.state, batch.action)\n        critic_loss = 0.5 * (q1 - target_q).square().mean() + 0.5 * (q2 - target_q).square().mean()\n\n        # ----- Repaired GFP actor paths -----\n        actor_action_raw = actor(batch.state, z)\n        with torch.no_grad():\n            flow_anchor = flow_sample(flow, batch.state, z, flow_steps)\n        distill_loss = (actor_action_raw - flow_anchor).square().mean()\n        actor_action = actor_action_raw.clamp(-1.0, 1.0)\n        # Current critic parameters are constants for actor loss, while dQ/da remains.\n        with frozen_parameters(critic):\n            q_actor = q_aggregate(critic, batch.state, actor_action, q_rule)\n        q_loss = -q_actor.mean()\n        lam = 1.0 / q_actor.detach().abs().mean().clamp_min(1e-6)\n        q_loss = lam * q_loss\n        actor_base_loss = alpha * distill_loss + q_loss\n\n        # ----- GFP value-aware behavior flow matching -----\n        x0 = torch.randn_like(batch.action)\n        t = torch.rand((batch_size, 1), device=device)\n        xt = (1.0 - t) * x0 + t * batch.action\n        velocity_target = batch.action - x0\n        velocity_pred = flow(batch.state, xt, t)\n        flow_error = (velocity_pred - velocity_target).square().mean(dim=-1)\n        if eta == 0.0:\n            guidance = torch.ones_like(flow_error)\n        else:\n            with torch.no_grad():\n                g_actor = None if guidance_rule in {"vabc", "flow"} else actor_action\n                g_flow = None if guidance_rule == "actor" else flow_anchor\n                q_ref = value_options(\n                    target_critic,\n                    batch.state,\n                    g_actor,\n                    g_flow,\n                    rule=guidance_rule,\n                    q_rule=q_rule,\n                )\n                q_data = q_aggregate(target_critic, batch.state, batch.action, q_rule)\n                logits = ((lam / eta) * (q_data - q_ref)).clamp(-60.0, 60.0)\n                guidance = torch.sigmoid(logits)\n        vabc_loss = (guidance.detach() * flow_error).mean()\n\n        # ----- CATS-GFP-R3-PATH -----\n        # PATH = Projected Antithetic Transition-tangent Homotopy.\n        tangent_cost = torch.zeros((), device=device)\n        excess_loss = torch.zeros((), device=device)\n        gate_mass = torch.zeros((), device=device)\n        residual_norm = torch.zeros((), device=device)\n        lcb_gap_mean = torch.zeros((), device=device)\n        probe_rms_mean = torch.zeros((), device=device)\n        batch_q75 = float("nan")\n        schedule_lambda = 0.0\n        lambda_eff = 0.0\n        grad_cos_before = float("nan")\n        grad_cos_after = float("nan")\n        aux_grad_norm_value = 0.0\n        projected_aux_grad_norm_value = 0.0\n        path_computed = False\n\n        # During calibration we measure the geometry but apply no intervention.\n        need_path_geometry = r3 and step >= calibration_start and (step < active_end or persistent_auxiliary)\n        if need_path_geometry:\n            path_computed = True\n            chaos_amp = probe.sample(batch_size, 1, device, batch.state.dtype).abs()\n            direction_std = ((batch.next_state - batch.state) / transition_rms.clamp_min(1e-3)).clamp(-3.0, 3.0)\n            direction_rms = direction_std.square().mean(dim=-1, keepdim=True).sqrt()\n            direction_std = direction_std / direction_rms.clamp_min(1.0)\n            normalized_probe = probe_radius * chaos_amp * direction_std\n            probe_delta = normalized_probe * transition_rms\n            probe_rms = normalized_probe.square().mean(dim=-1).sqrt()\n            probe_rms_mean = probe_rms.mean()\n            plus_state = batch.state + probe_delta\n            minus_state = batch.state - probe_delta\n\n            plus_actor_raw = actor(plus_state, z)\n            minus_actor_raw = actor(minus_state, z)\n            with torch.no_grad():\n                plus_flow = flow_sample(flow, plus_state, z, flow_steps)\n                minus_flow = flow_sample(flow, minus_state, z, flow_steps)\n            residual_clean = actor_action_raw - flow_anchor\n            residual_plus = plus_actor_raw - plus_flow\n            residual_minus = minus_actor_raw - minus_flow\n            central_delta = residual_plus - residual_minus\n\n            denominator = 4.0 * normalized_probe.square().mean(dim=-1).clamp_min(probe_denominator_floor ** 2)\n            per_cost = central_delta.square().mean(dim=-1) / denominator\n            per_residual_norm = torch.stack([\n                residual_clean.square().mean(dim=-1).sqrt(),\n                residual_plus.square().mean(dim=-1).sqrt(),\n                residual_minus.square().mean(dim=-1).sqrt(),\n            ], dim=0).max(dim=0).values\n            residual_norm = per_residual_norm.mean()\n            batch_q75 = float(torch.quantile(per_cost.detach(), 0.75))\n\n            # Calibrate a frozen task-relative tangent budget under pure GFP updates.\n            if calibration_start <= step <= calibration_end and not budget_frozen:\n                if not math.isfinite(budget_q75_ema):\n                    budget_q75_ema = batch_q75\n                else:\n                    budget_q75_ema = budget_ema_beta * budget_q75_ema + (1.0 - budget_ema_beta) * batch_q75\n                if step >= calibration_end:\n                    tangent_budget = max(min_tangent_budget, budget_multiplier * budget_q75_ema)\n                    budget_frozen = True\n                    print(\n                        f"[PATH budget frozen] step={step} q75_ema={budget_q75_ema:.6g} budget={tangent_budget:.6g}",\n                        flush=True,\n                    )\n\n            if step > calibration_end and not budget_frozen:\n                fallback = batch_q75 if math.isfinite(batch_q75) else min_tangent_budget\n                tangent_budget = max(min_tangent_budget, budget_multiplier * fallback)\n                budget_q75_ema = fallback\n                budget_frozen = True\n\n            with torch.no_grad():\n                # Twin-critic lower confidence gap across clean and antithetic states.\n                state_action_pairs = [\n                    (batch.state, actor_action, flow_anchor),\n                    (plus_state, plus_actor_raw.clamp(-1.0, 1.0), plus_flow),\n                    (minus_state, minus_actor_raw.clamp(-1.0, 1.0), minus_flow),\n                ]\n                gaps = []\n                for st, aa, fa in state_action_pairs:\n                    q1a, q2a = target_critic(st, aa)\n                    q1f, q2f = target_critic(st, fa)\n                    gaps.extend([q1a - q1f, q2a - q2f])\n                lcb_gap = torch.stack(gaps, dim=0).min(dim=0).values\n                lcb_gap_mean = lcb_gap.mean()\n                safety_gate = torch.sigmoid((lcb_gap + safety_margin) / max(safety_temperature, 1e-6))\n                radius_gate = (per_residual_norm <= max_residual).float()\n                transition_gate = (probe_rms >= min_probe_rms).float()\n                gate = safety_gate * radius_gate * transition_gate\n            gate_mass = gate.mean()\n            tangent_cost = (gate * per_cost).sum() / gate.sum().clamp_min(1.0)\n\n            if budget_frozen and math.isfinite(tangent_budget):\n                relative_excess = F.relu(per_cost / max(tangent_budget, min_tangent_budget) - 1.0)\n                per_penalty = _huber_positive(relative_excess)\n                excess_loss = (gate * per_penalty).sum() / gate.sum().clamp_min(1.0)\n            if persistent_auxiliary:\n                schedule_lambda = lambda_max if step > calibration_end else 0.0\n            else:\n                schedule_lambda = _pulse_schedule(step, calibration_end, active_end, lambda_max)\n\n        # ----- Exact module-wise gradient paths -----\n        optimizer.zero_grad(set_to_none=True)\n        critic_grads = autograd_list(critic_loss, critic_params)\n        flow_grads = autograd_list(vabc_loss, flow_params)\n\n        active_aux = bool(\n            r3\n            and path_computed\n            and budget_frozen\n            and schedule_lambda > 0.0\n            and float(gate_mass.detach()) >= min_gate_mass\n            and float(excess_loss.detach()) > 0.0\n        )\n        actor_base_grads = autograd_list(actor_base_loss, actor_params, retain_graph=active_aux)\n        actor_base_norm = list_norm(actor_base_grads)\n        actor_update_grads = actor_base_grads\n\n        if active_aux:\n            aux_grads = autograd_list(excess_loss, actor_params)\n            aux_norm = list_norm(aux_grads)\n            aux_grad_norm_value = float(aux_norm.detach())\n            base_sq = list_dot(actor_base_grads, actor_base_grads)\n            dot_before = list_dot(aux_grads, actor_base_grads)\n            denom_cos = (aux_norm * actor_base_norm).clamp_min(1e-12)\n            grad_cos_before = float((dot_before / denom_cos).detach())\n            # Exact half-space projection when the GFP actor gradient is nonzero.\n            # If the base gradient is numerically zero, suppress the auxiliary update\n            # so the method falls back to GFP rather than dividing by a tiny norm.\n            if not projection_enabled:\n                projected = aux_grads\n            elif float(base_sq.detach()) > 1e-20:\n                coeff = torch.minimum(dot_before, torch.zeros_like(dot_before)) / base_sq\n                projected = [h - coeff * g for h, g in zip(aux_grads, actor_base_grads)]\n            else:\n                projected = [torch.zeros_like(h) for h in aux_grads]\n            projected_norm = list_norm(projected)\n            projected_aux_grad_norm_value = float(projected_norm.detach())\n            grad_cos_after = float((list_dot(projected, actor_base_grads) / (projected_norm * actor_base_norm).clamp_min(1e-12)).detach())\n            norm_cap = auxiliary_grad_ratio * float(actor_base_norm.detach()) / max(float(projected_norm.detach()), 1e-12)\n            lambda_eff = min(schedule_lambda, norm_cap)\n            actor_update_grads = [g + float(lambda_eff) * h for g, h in zip(actor_base_grads, projected)]\n\n        assign_gradients(critic_params, critic_grads)\n        assign_gradients(flow_params, flow_grads)\n        assign_gradients(actor_params, actor_update_grads)\n        actor_gn = float(list_norm(actor_update_grads).detach())\n        actor_base_gn = float(actor_base_norm.detach())\n        flow_gn = float(list_norm(flow_grads).detach())\n        critic_gn = float(list_norm(critic_grads).detach())\n        optimizer.step()\n        soft_update(critic, target_critic, tau)\n\n        total_loss_for_log = critic_loss.detach() + vabc_loss.detach() + actor_base_loss.detach() + float(lambda_eff) * excess_loss.detach()\n        if step % log_every == 0 or step == steps:\n            last_diag = {\n                "step": step,\n                "total_loss_proxy": float(total_loss_for_log),\n                "critic_loss": float(critic_loss.detach()),\n                "vabc_loss": float(vabc_loss.detach()),\n                "distill_loss": float(distill_loss.detach()),\n                "q_loss": float(q_loss.detach()),\n                "q_scale_lambda": float(lam.detach()),\n                "guidance_mean": float(guidance.mean()),\n                "guidance_gt_05": float((guidance > 0.5).float().mean()),\n                "actor_base_grad_norm": actor_base_gn,\n                "actor_update_grad_norm": actor_gn,\n                "flow_grad_norm": flow_gn,\n                "critic_grad_norm": critic_gn,\n                "path_tangent_cost": float(tangent_cost.detach()),\n                "path_excess_loss": float(excess_loss.detach()),\n                "path_budget_q75_batch": batch_q75,\n                "path_budget_q75_ema": budget_q75_ema,\n                "path_tangent_budget": tangent_budget,\n                "path_budget_frozen": int(budget_frozen),\n                "path_gate_mass": float(gate_mass.detach()),\n                "path_schedule_lambda": float(schedule_lambda),\n                "path_lambda_eff": float(lambda_eff),\n                "path_aux_grad_norm": aux_grad_norm_value,\n                "path_projected_aux_grad_norm": projected_aux_grad_norm_value,\n                "path_grad_cos_before": grad_cos_before,\n                "path_grad_cos_after": grad_cos_after,\n                "path_residual_norm": float(residual_norm.detach()),\n                "path_lcb_gap": float(lcb_gap_mean.detach()),\n                "path_probe_rms": float(probe_rms_mean.detach()),\n                "elapsed_sec": time.time() - t0,\n            }\n            append_csv(train_metrics_path, last_diag)\n            print("[train]", name, method, last_diag, flush=True)\n\n        if step % eval_every == 0 or step == steps:\n            ep_count = eval_episodes if step < steps else final_eval_episodes\n            ev = evaluate_policy(env, ds, actor, episodes=ep_count, seed=eval_seed, device=device)\n            row = {\n                "step": step,\n                "dataset": name,\n                "suite": suite,\n                "method": method,\n                "seed": seed,\n                "eval_episodes": ep_count,\n                **ev,\n                "elapsed_sec": time.time() - t0,\n            }\n            append_csv(metrics_path, row)\n            last_eval_result = dict(ev)\n            print("[eval]", row, flush=True)\n\n        if step % save_every == 0 or step == steps:\n            save_checkpoint(step)\n\n    final_eval = last_eval_result\n    if final_eval is None:\n        final_eval = evaluate_policy(env, ds, actor, episodes=final_eval_episodes, seed=eval_seed, device=device)\n    try:\n        from importlib import metadata as _metadata\n        ogbench_version = _metadata.version("ogbench") if suite == "ogbench" else None\n    except Exception:\n        ogbench_version = None\n    result = {\n        "dataset": name,\n        "suite": suite,\n        "method": method,\n        "seed": seed,\n        "steps": steps,\n        "eval_episodes": final_eval_episodes,\n        **final_eval,\n        "elapsed_sec": time.time() - t0,\n        "device": str(device),\n        "ogbench_version": ogbench_version,\n        "ogbench_protocol": "official GFP ogbench<1.2" if suite == "ogbench" else None,\n        "dataset_size": ds.size,\n        "obs_dim": state_dim,\n        "action_dim": action_dim,\n        "profile": profile,\n        "gradient_path_repair": {\n            "critic_from_bellman_only": True,\n            "flow_from_vabc_only": True,\n            "actor_q_uses_frozen_current_critic": True,\n            "shared_global_gradient_clip": False,\n        },\n        "r3_path_config": {\n            "enabled": r3,\n            "calibration_start": calibration_start,\n            "calibration_end": calibration_end,\n            "active_end": active_end,\n            "probe_radius": probe_radius,\n            "budget_multiplier": budget_multiplier,\n            "final_tangent_budget": tangent_budget,\n            "lambda_max": lambda_max,\n            "auxiliary_grad_ratio": auxiliary_grad_ratio,\n            "safety_margin": safety_margin,\n            "safety_temperature": safety_temperature,\n            "max_residual": max_residual,\n            "projection_enabled": projection_enabled,\n            "persistent_auxiliary": persistent_auxiliary,\n            "probe_kind": ("gaussian" if method.endswith("gaussian_probe") else "uniform" if method.endswith("uniform_probe") else "lorenz"),\n        },\n        "last_training": last_diag,\n    }\n    atomic_json(output / "final.json", result)\n    try:\n        env.close()\n    except Exception:\n        pass\n    print("[done]", result, flush=True)\n    return result\n\n\ndef preflight_dataset(name: str) -> None:\n    env, raw, _val, suite = load_raw_dataset(name, seed=0)\n    try:\n        n = len(raw["observations"])\n        print(f"[dataset preflight] PASS name={name} suite={suite} transitions={n}", flush=True)\n    finally:\n        try:\n            env.close()\n        except Exception:\n            pass\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--job")\n    parser.add_argument("--preflight-dataset")\n    args = parser.parse_args()\n    if args.preflight_dataset:\n        preflight_dataset(args.preflight_dataset)\n        return\n    if not args.job:\n        parser.error("--job is required unless --preflight-dataset is used")\n    train(json.loads(Path(args.job).read_text()))\n\n\nif __name__ == "__main__":\n    main()\n'

SUPPORT_DF = pd.DataFrame(BACKBONE_SUPPORT)
SUPPORT_MAP = {r["dataset"]: r for r in BACKBONE_SUPPORT}
for d in DATASETS:
    if d not in GFP_PROFILES:
        raise KeyError(f"No embedded GFP profile for {d}")

D4RL_REQUESTED = any(("singletask" not in d and not d.startswith("toy-")) for d in DATASETS)
OGBENCH_REQUESTED = any("singletask" in d for d in DATASETS)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUNTIME_DIR = OUTPUT_ROOT / "_runtime"
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
RUNNER_PATH = RUNTIME_DIR / "path_gfp_ablation_embedded_runner.py"
DATASET_LOCK_DIR = RUNTIME_DIR / "dataset_locks"
DATASET_LOCK_DIR.mkdir(parents=True, exist_ok=True)


def _ensure_import(module: str, install_args=None):
    if importlib.util.find_spec(module) is not None:
        return
    if not AUTO_INSTALL_MISSING:
        raise ImportError(f"Missing package {module!r}; install it or enable AUTO_INSTALL_MISSING.")
    args = list(install_args or [module])
    print(f"[install] {module}: {' '.join(args)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


def _dist_version(name: str):
    try:
        from importlib import metadata
        return metadata.version(name)
    except Exception:
        return None


def _version_tuple(version):
    nums = [int(x) for x in re.findall(r"\d+", str(version or ""))[:3]]
    while len(nums) < 3:
        nums.append(0)
    return tuple(nums)


def _major_version(version):
    return _version_tuple(version)[0] if version else None


# D4RL children get an isolated Cython/NumPy compatibility overlay. The
# Jupyter and OGBench environment is not downgraded.
LEGACY_D4RL_COMPAT_DIR = RUNTIME_DIR / "legacy_d4rl_compat"
LEGACY_D4RL_CYTHON = "0.29.37"
LEGACY_D4RL_NUMPY = "1.26.4"
LEGACY_D4RL_MARKER = LEGACY_D4RL_COMPAT_DIR / "cats_compat.json"


def _prepare_legacy_d4rl_compat_overlay():
    if not D4RL_REQUESTED:
        return
    base_numpy = _dist_version("numpy")
    need_numpy_1x = (_major_version(base_numpy) or 0) >= 2
    desired = {
        "python": f"{sys.version_info.major}.{sys.version_info.minor}",
        "cython": LEGACY_D4RL_CYTHON,
        "numpy": LEGACY_D4RL_NUMPY if need_numpy_1x else "base<2",
    }
    current = None
    if LEGACY_D4RL_MARKER.exists():
        try:
            current = json.loads(LEGACY_D4RL_MARKER.read_text())
        except Exception:
            pass
    cython_present = (LEGACY_D4RL_COMPAT_DIR / "Cython").exists()
    numpy_present = any(LEGACY_D4RL_COMPAT_DIR.glob("numpy*")) if need_numpy_1x else True
    if current == desired and cython_present and numpy_present:
        print(f"[d4rl compat] ready: {LEGACY_D4RL_COMPAT_DIR}")
        return
    if not AUTO_INSTALL_MISSING:
        raise RuntimeError("Legacy D4RL compatibility overlay is missing.")
    print("[d4rl compat] preparing isolated legacy runtime overlay")
    shutil.rmtree(LEGACY_D4RL_COMPAT_DIR, ignore_errors=True)
    LEGACY_D4RL_COMPAT_DIR.mkdir(parents=True, exist_ok=True)
    base_cmd = [sys.executable, "-m", "pip", "install", "-q", "--target", str(LEGACY_D4RL_COMPAT_DIR), "--no-deps"]
    subprocess.check_call(base_cmd + [f"Cython=={LEGACY_D4RL_CYTHON}"])
    if need_numpy_1x:
        subprocess.check_call(base_cmd + [f"numpy=={LEGACY_D4RL_NUMPY}"])
    LEGACY_D4RL_MARKER.write_text(json.dumps(desired, indent=2, sort_keys=True))


def _legacy_d4rl_subprocess_env():
    env = os.environ.copy()
    env["MUJOCO_GL"] = MUJOCO_GL
    env["CUDA_VISIBLE_DEVICES"] = ""
    old = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(LEGACY_D4RL_COMPAT_DIR) + (os.pathsep + old if old else "")
    env["CATS_DATASET_LOCK_DIR"] = str(DATASET_LOCK_DIR)
    return env


def _preflight_legacy_d4rl():
    if not D4RL_REQUESTED:
        return
    first_dataset = next(d for d in DATASETS if "singletask" not in d and not d.startswith("toy-"))
    code = r'''import importlib.metadata as md
import sys

def ver(name):
    try: return md.version(name)
    except Exception: return 'unknown'
print('[d4rl preflight child] Python', sys.version.split()[0])
print('[d4rl preflight child] Cython', ver('Cython'))
print('[d4rl preflight child] NumPy', ver('numpy'))
import mujoco_py
print('[d4rl preflight child] mujoco_py import OK')
import gym
import d4rl
name = sys.argv[1]
env = gym.make(name)
try:
    env.reset()
    print('[d4rl preflight child] env OK:', name, env.observation_space, env.action_space)
finally:
    try: env.close()
    except Exception: pass
'''
    print(f"[d4rl preflight] compiling/importing legacy runtime once with {first_dataset}")
    proc = subprocess.run(
        [sys.executable, "-u", "-c", code, first_dataset],
        env=_legacy_d4rl_subprocess_env(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(proc.stdout.rstrip())
    if proc.returncode != 0:
        raise RuntimeError("Legacy D4RL runtime preflight failed before experiment launch.")
    print("[d4rl preflight] PASS")


_ensure_import("numpy", ["numpy"])
_ensure_import("torch", ["torch"])
_ensure_import("pandas", ["pandas"])
_ensure_import("tabulate", ["tabulate"])
if D4RL_REQUESTED:
    _prepare_legacy_d4rl_compat_overlay()
    _ensure_import("d4rl", ["git+https://github.com/Farama-Foundation/D4RL@master#egg=d4rl"])
if OGBENCH_REQUESTED:
    _ensure_import("ogbench", ["ogbench<1.2"])
    current_ogbench = _dist_version("ogbench")
    if _version_tuple(current_ogbench) >= (1, 2, 0):
        if not AUTO_INSTALL_MISSING:
            raise RuntimeError(f"Official GFP protocol requires ogbench<1.2; found {current_ogbench}")
        print(f"[install] aligning OGBench to official GFP protocol: found {current_ogbench}, installing ogbench<1.2")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "ogbench<1.2"])

RUNNER_PATH.write_text(RUNNER_SOURCE)
py_compile.compile(str(RUNNER_PATH), doraise=True)
SUPPORT_DF.to_csv(OUTPUT_ROOT / "backbone_support_matrix.csv", index=False)

print("CATS-GFP-R3-PATH dual-cell iteration pipeline configured.")
print("  stage:", STAGE)
print("  datasets:", len(DATASETS))
print("  methods:", METHODS)
print("  steps:", STEPS)
print("  seed(s):", SEEDS)
print("  GPUs:", GPUS)
print("  output:", OUTPUT_ROOT)
print("  runner compile: PASS")


In [ ]:
# ================================================================
# CELL 2 / 2 — dual-GPU focused ablations, paired statistics, and claim gate
# ================================================================
from collections import deque
import json, os, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

os.environ.setdefault("MUJOCO_GL", MUJOCO_GL)


def _suite(name):
    return "ogbench" if "singletask" in name else "d4rl"


def _gpu_inventory():
    try:
        return [x.strip() for x in subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader,nounits"], text=True
        ).splitlines() if x.strip()]
    except Exception:
        return []


def _worker_env(gpu, suite):
    env = os.environ.copy()
    env["MUJOCO_GL"] = MUJOCO_GL
    env["CATS_DATASET_LOCK_DIR"] = str(DATASET_LOCK_DIR)
    env["CUDA_VISIBLE_DEVICES"] = str(gpu) if gpu >= 0 else ""
    if suite == "d4rl":
        old = env.get("PYTHONPATH", "")
        env["PYTHONPATH"] = str(LEGACY_D4RL_COMPAT_DIR) + (os.pathsep + old if old else "")
    return env


inventory = _gpu_inventory()
print("GPU inventory:")
for row in inventory:
    print("  ", row)
if REQUIRE_GPU and not inventory:
    raise RuntimeError("No NVIDIA GPU found while REQUIRE_GPU=True.")

if D4RL_REQUESTED:
    _preflight_legacy_d4rl()
if OGBENCH_REQUESTED and SERIAL_DATASET_PREFLIGHT:
    for dataset in [d for d in DATASETS if _suite(d) == "ogbench"]:
        proc = subprocess.run(
            [sys.executable, "-u", str(RUNNER_PATH), "--preflight-dataset", dataset],
            env=_worker_env(-1, "ogbench"), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        print(proc.stdout.rstrip())
        if proc.returncode != 0:
            raise RuntimeError(f"OGBench preflight failed for {dataset}")

BASE_JOB = dict(
    steps=int(STEPS), eval_every=int(EVAL_EVERY), eval_episodes=int(EVAL_EPISODES),
    final_eval_episodes=int(FINAL_EVAL_EPISODES), eval_seed=int(EVAL_SEED),
    log_every=int(LOG_EVERY), save_every=int(SAVE_EVERY), require_gpu=bool(REQUIRE_GPU),
    resume=bool(RESUME), hidden=int(HIDDEN), depth=int(DEPTH), time_dim=int(TIME_DIM),
    lr=float(LR), tau=float(TAU), probe_radius=float(PROBE_RADIUS), probe_steps=int(PROBE_STEPS),
    probe_burn_in=int(PROBE_BURN_IN), calibration_start=int(CALIBRATION_START),
    calibration_end=int(CALIBRATION_END), active_end=int(ACTIVE_END),
    budget_multiplier=float(BUDGET_MULTIPLIER), budget_ema_beta=float(BUDGET_EMA_BETA),
    min_tangent_budget=float(MIN_TANGENT_BUDGET), lambda_max=float(LAMBDA_MAX),
    auxiliary_grad_ratio=float(AUXILIARY_GRAD_RATIO), safety_margin=float(SAFETY_MARGIN),
    safety_temperature=float(SAFETY_TEMPERATURE), max_residual=float(MAX_RESIDUAL),
    min_probe_rms=float(MIN_PROBE_RMS), min_gate_mass=float(MIN_GATE_MASS),
    probe_denominator_floor=float(PROBE_DENOMINATOR_FLOOR),
)


def _completed(path, method, seed):
    try:
        x = json.loads(Path(path).read_text())
        return int(x.get("steps", -1)) == int(STEPS) and x.get("method") == method and int(x.get("seed", -1)) == int(seed)
    except Exception:
        return False

jobs = []
for dataset in DATASETS:
    for seed in SEEDS:
        for method in METHODS:
            run_dir = OUTPUT_ROOT / _suite(dataset) / dataset / method / f"seed_{seed:02d}"
            run_dir.mkdir(parents=True, exist_ok=True)
            job = dict(BASE_JOB, dataset=dataset, method=method, seed=int(seed),
                       eval_seed=int(EVAL_SEED + 1000 * int(seed)), output=str(run_dir),
                       profile=dict(GFP_PROFILES[dataset]), formal_gate_weight=1.0)
            jp = run_dir / "job.json"
            jp.write_text(json.dumps(job, indent=2, sort_keys=True))
            if not (RESUME and _completed(run_dir / "final.json", method, seed)):
                jobs.append(dict(dataset=dataset, method=method, seed=seed, suite=_suite(dataset), run_dir=run_dir, job_path=jp, attempt=0))

print(f"Jobs to execute now: {len(jobs)}")
print(f"Matrix: {len(DATASETS)} datasets x {len(METHODS)} methods x {len(SEEDS)} seeds")
pending = deque(jobs)
slots = [(int(g), i) for g in GPUS for i in range(int(WORKERS_PER_GPU))] or [(-1, 0)]
running, failures = {}, []


def _launch(slot, job):
    gpu, local_slot = slot
    log = job["run_dir"] / "stdout.log"
    h = log.open("a", buffering=1)
    print(f"[launch] GPU={gpu} slot={local_slot} :: {job['dataset']} :: {job['method']} :: seed={job['seed']}")
    p = subprocess.Popen([sys.executable, "-u", str(RUNNER_PATH), "--job", str(job["job_path"])],
                         env=_worker_env(gpu, job["suite"]), stdout=h, stderr=subprocess.STDOUT, text=True)
    return {**job, "process": p, "handle": h, "log": log}

while pending or running:
    for slot in slots:
        if slot not in running and pending:
            running[slot] = _launch(slot, pending.popleft())
    if not running:
        break
    time.sleep(1)
    done = []
    for slot, a in list(running.items()):
        code = a["process"].poll()
        if code is None:
            continue
        a["handle"].close(); done.append(slot)
        if code == 0:
            print("[done]", a["dataset"], a["method"], a["seed"])
        else:
            tail = "\n".join(a["log"].read_text(errors="replace").splitlines()[-80:])
            print("[FAILED]", a["dataset"], a["method"], a["seed"], "\n", tail)
            retry = {k:v for k,v in a.items() if k not in {"process","handle","log"}}
            retry["attempt"] += 1
            if retry["attempt"] <= MAX_JOB_RETRIES:
                pending.append(retry)
            else:
                failures.append(dict(dataset=a["dataset"], method=a["method"], seed=a["seed"], log=str(a["log"])))
    for slot in done:
        running.pop(slot, None)

(OUTPUT_ROOT / "technical_failures.json").write_text(json.dumps(failures, indent=2))

# Aggregate final and checkpoint metrics.
runs, curves, diags = [], [], []
for dataset in DATASETS:
    for method in METHODS:
        for seed in SEEDS:
            rd = OUTPUT_ROOT / _suite(dataset) / dataset / method / f"seed_{seed:02d}"
            if (rd/"final.json").exists():
                x=json.loads((rd/"final.json").read_text()); x["run_dir"]=str(rd); runs.append(x)
            if (rd/"metrics.csv").exists():
                f=pd.read_csv(rd/"metrics.csv"); f["run_dir"]=str(rd); curves.append(f)
            if (rd/"training_metrics.csv").exists():
                f=pd.read_csv(rd/"training_metrics.csv"); f["dataset"]=dataset; f["method"]=method; f["seed"]=seed; diags.append(f)
run_df=pd.DataFrame(runs)
curve_df=pd.concat(curves,ignore_index=True) if curves else pd.DataFrame()
diag_df=pd.concat(diags,ignore_index=True) if diags else pd.DataFrame()
if len(run_df): run_df=run_df.sort_values(["dataset","method","seed","steps"]).drop_duplicates(["dataset","method","seed"],keep="last")
if len(curve_df): curve_df=curve_df.sort_values(["dataset","method","seed","step","elapsed_sec"]).drop_duplicates(["dataset","method","seed","step"],keep="last")
run_df.to_csv(OUTPUT_ROOT/"all_runs.csv",index=False)
curve_df.to_csv(OUTPUT_ROOT/"all_metrics.csv",index=False)
diag_df.to_csv(OUTPUT_ROOT/"training_diagnostics.csv",index=False)

baseline="cats_gfp_r3_path"
rows=[]
for dataset in DATASETS:
    for seed in SEEDS:
        b=run_df[(run_df.dataset==dataset)&(run_df.method==baseline)&(run_df.seed.astype(int)==int(seed))]
        bc=curve_df[(curve_df.dataset==dataset)&(curve_df.method==baseline)&(curve_df.seed.astype(int)==int(seed))].sort_values("step")
        if len(b)!=1 or not len(bc):
            continue
        bfinal=float(b.iloc[0].score); blate=float(bc.tail(min(4,len(bc))).score.mean())
        for method in METHODS:
            if method==baseline: continue
            r=run_df[(run_df.dataset==dataset)&(run_df.method==method)&(run_df.seed.astype(int)==int(seed))]
            rc=curve_df[(curve_df.dataset==dataset)&(curve_df.method==method)&(curve_df.seed.astype(int)==int(seed))].sort_values("step")
            if len(r)!=1 or not len(rc): continue
            rfinal=float(r.iloc[0].score); rlate=float(rc.tail(min(4,len(rc))).score.mean())
            rows.append(dict(dataset=dataset,seed=seed,method=method,baseline=baseline,
                             baseline_final=bfinal,method_final=rfinal,final_delta=rfinal-bfinal,
                             baseline_late=blate,method_late=rlate,late_delta=rlate-blate,
                             runtime_ratio=float(r.iloc[0].elapsed_sec)/max(float(b.iloc[0].elapsed_sec),1e-9)))
paired=pd.DataFrame(rows)
paired.to_csv(OUTPUT_ROOT/"ablation_seed_results.csv",index=False)
summary=(paired.groupby(["dataset","method"],as_index=False)
         .agg(seeds=("seed","count"),final_delta_mean=("final_delta","mean"),final_delta_std=("final_delta","std"),
              late_delta_mean=("late_delta","mean"),runtime_ratio=("runtime_ratio","median"))) if len(paired) else pd.DataFrame()
summary.to_csv(OUTPUT_ROOT/"ablation_task_summary.csv",index=False)
variant=(paired.groupby("method",as_index=False)
         .agg(final_delta_mean=("final_delta","mean"),final_delta_median=("final_delta","median"),
              late_delta_mean=("late_delta","mean"),late_delta_median=("late_delta","median"),
              runtime_ratio=("runtime_ratio","median"))) if len(paired) else pd.DataFrame()
variant.to_csv(OUTPUT_ROOT/"ablation_variant_summary.csv",index=False)

# Claim gate. This does not decide whether the main method is useful; it decides
# which components may be claimed in the paper.
def med(method,col):
    x=paired.loc[paired.method==method,col].to_numpy(float)
    return float(np.median(x)) if len(x) else np.nan

probe_methods={
    "lorenz": baseline,
    "gaussian": "cats_gfp_r3_gaussian_probe",
    "uniform": "cats_gfp_r3_uniform_probe",
}
probe_score={k:(0.0 if m==baseline else med(m,"final_delta")) for k,m in probe_methods.items()}
best_probe=max(probe_score,key=lambda k: (-1e18 if not np.isfinite(probe_score[k]) else probe_score[k]))
projection_supported = med("cats_gfp_r3_no_projection","final_delta") <= 0.0 or med("cats_gfp_r3_no_projection","late_delta") <= 0.0
homotopy_supported = med("cats_gfp_r3_no_homotopy","late_delta") <= 0.0
if failures:
    decision="FAIL_TECHNICAL"
elif projection_supported and homotopy_supported and best_probe=="lorenz":
    decision="FULL_PATH_DESIGN_SUPPORTED"
elif projection_supported and homotopy_supported:
    decision="PATH_SUPPORTED_BUT_NOT_CHAOS_SPECIFIC"
else:
    decision="MECHANISM_CLAIM_MUST_BE_NARROWED"

gate=dict(decision=decision,projection_supported=bool(projection_supported),homotopy_supported=bool(homotopy_supported),
          probe_score=probe_score,best_probe=best_probe,technical_failures=len(failures))
(OUTPUT_ROOT/"ablation_gate_decision.json").write_text(json.dumps(gate,indent=2,sort_keys=True))
lines=["# PATH-GFP focused ablation report","",f"- Decision: **{decision}**",
       f"- Projection supported: `{projection_supported}`",f"- Homotopy/late fallback supported: `{homotopy_supported}`",
       f"- Best probe family under this small matrix: `{best_probe}`","", "## Variant summary","", variant.to_markdown(index=False) if len(variant) else "No complete results.",
       "", "## Task summary","", summary.to_markdown(index=False) if len(summary) else "No complete results.",
       "", "## Interpretation","",
       "If Gaussian or uniform probes match/exceed Lorenz, the paper must present PATH as transition-aligned structured probing and treat Lorenz only as one instantiation. Do not claim a chaos-specific advantage."]
(OUTPUT_ROOT/"ablation_gate_report.md").write_text("\n".join(lines))
print("Decision:",decision,"Artifacts:",OUTPUT_ROOT)
display(variant); display(summary)
